In [1]:
import time 
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from PIL import Image
import torch 
import torch.nn as nn
from torchvision.models import vit_h_14
from torchvision import transforms
import os
import joblib
from sklearn.tree import DecisionTreeClassifier


main_model_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])


feature_description = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "image_name": tf.io.FixedLenFeature([], tf.string)
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torch.load('/kaggle/input/resnet50_70_512x512/pytorch/default/1/Resnet50_70_512x512.pth')
model.to(device)

path = "/kaggle/input/cassava-leaf-disease-classification/test_tfrecords"
tfrecs = os.listdir(path)

combined_probs = []
prediction = []
image_ids = []

for tfrec in tfrecs:
    raw_dataset = tf.data.TFRecordDataset(f"{path}/{tfrec}")

    for raw_record in raw_dataset:
        
        parsed_record = tf.io.parse_single_example(raw_record, feature_description)

        image_bytes = parsed_record["image"]
        image = tf.io.decode_jpeg(image_bytes)

        image_name = parsed_record["image_name"].numpy().decode("utf-8")
        image_ids.append(image_name)
        image_np = image.numpy()
        image_pil = Image.fromarray(image_np)

        image_tensor_py = main_model_preprocess(image_pil)
        image_tensor_py = image_tensor_py.unsqueeze(0).to(device)
        with torch.no_grad():
            output_py = model(image_tensor_py).cpu()
            del image_tensor_py
            torch.cuda.empty_cache()
        prediction.append(np.argmax(output_py.numpy()[0]))

2026-04-28 05:46:03.696033: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777355163.709940      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777355163.714316      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 05:46:03.730704: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/resnet50_70_512x512/pytorch/default/1/Resnet50_70_512x512.pth'

In [2]:
submission = pd.DataFrame({
    'image_id': image_ids,
    'label': prediction
})
submission.to_csv('submission.csv', index=False)

NameError: name 'image_ids' is not defined